# 01_Exploracion_Archivos_Originales.ipynb

In [1]:
# https://grok.com/c/7d8e98ba-2c82-4953-8860-d4ba78d912ef?rid=e83c8206-5d55-4e6c-9fa4-5bf7d780b899

# 01 – Exploración de los Archivos Originales

**Proyecto:** C2011-13 Wind Farm  
**Objetivo de este notebook:** Entender de forma clara y ordenada qué información contienen los tres archivos originales del proyecto, antes de hacer ninguna transformación.

Este notebook está escrito como un tutorial. La idea es que cualquier persona pueda seguir el razonamiento sin necesidad de ser técnico.

## ¿Qué archivos tenemos?

En la carpeta `Data_Original` se encuentran tres archivos:

| Archivo | Tipo | Descripción breve |
|---------|------|-------------------|
| `C2011-13 Wind Farm.xlsx` | Excel | Contiene la información principal del proyecto (actividades, costes, seguimiento...) |
| `C2011-13 Wind Farm.p2x` | ProTrack | Archivo nativo del software de gestión de proyectos ProTrack |
| `C2011-13 Wind Farm.pdf` | PDF | Ficha resumen del proyecto (Project Card) |

Empezaremos por el archivo Excel, que es el más rico en información estructurada.

# Setup

In [6]:
from pathlib import Path
import openpyxl

# Rutas
BASE_DIR = Path.cwd()
DATA_ORIGINAL = BASE_DIR / "Data_Original"
EXCEL_FILE = DATA_ORIGINAL / "C2011-13 Wind Farm.xlsx"

print("Directorio de trabajo:", BASE_DIR)
print("Carpeta Data_Original existe:", DATA_ORIGINAL.exists())
print("Archivo Excel existe:", EXCEL_FILE.exists())

# Listar hojas del Excel (Código)

In [8]:
wb = openpyxl.load_workbook(EXCEL_FILE, data_only=True)

print(f"Número total de hojas: {len(wb.sheetnames)}\n")
print("Listado completo de hojas:")
print("-" * 50)

for i, name in enumerate(wb.sheetnames, 1):
    print(f"{i:3d}. {name}")

# Interpretación de las hojas (Markdown)

## ¿Qué nos dice este listado?

El archivo Excel contiene muchas hojas. A simple vista se pueden agrupar en varios bloques:

1. **Baseline Schedule** → Planificación inicial del proyecto (actividades, duraciones, costes previstos...)
2. **Gantt chart** y **Resources** → Información de planificación y recursos
3. **Risk Analysis** → Análisis de riesgos
4. **Project Control - TP1** y luego **TP2 ... TP120** → 120 periodos de seguimiento del proyecto
5. **Tracking Overview** → Resumen de todos los periodos de seguimiento
6. Hojas de indicadores (CPI, SPI, etc.)

La parte más valiosa para construir un sistema de seguimiento son:
- La hoja de **Baseline Schedule** (definición de actividades)
- Las hojas de **Tracking Periods** (evolución real del proyecto)
- La hoja **Tracking Overview** (visión agregada del rendimiento)

# Explorar Baseline Schedule

In [12]:
ws = wb["Baseline Schedule"]

print("=== Hoja: Baseline Schedule ===")
print(f"Dimensiones aproximadas: {ws.dimensions}")
print()

print("Cabeceras (fila 2):")
for col in range(1, 18):
    value = ws.cell(2, col).value
    if value is not None:
        print(f"  Columna {col:2d}: {value}")

# Ver primeras actividades

In [11]:
print("Primeras actividades del proyecto:\n")

for row in range(3, 13):
    id_ = ws.cell(row, 1).value
    name = ws.cell(row, 2).value
    wbs = ws.cell(row, 3).value
    duration = ws.cell(row, 8).value
    cost = ws.cell(row, 14).value
    
    print(f"ID {id_} | WBS {wbs} | {name} | Duración: {duration} | Coste: {cost}")

# Explorar Tracking Overview

In [14]:
ws_track = wb["Tracking Overview"]

print("=== Hoja: Tracking Overview ===")
print()

print("Cabeceras principales (fila 2):")
for col in range(1, 20):
    value = ws_track.cell(2, col).value
    if value is not None:
        print(f"  Columna {col:2d}: {value}")

# Número de periodos de seguimiento

In [16]:
# Contamos cuántos periodos de seguimiento hay
periodos = 0
for row in range(3, ws_track.max_row + 1):
    if ws_track.cell(row, 1).value is not None:
        periodos += 1

print(f"Número de Tracking Periods encontrados: {periodos}")

## Conclusiones de esta primera exploración

Después de revisar el archivo Excel original, podemos sacar estas conclusiones claras:

1. **El proyecto está bien documentado** en cuanto a planificación (Baseline Schedule).
2. Existe un **seguimiento muy detallado** a lo largo del tiempo (120 Tracking Periods).
3. Hay información de **rendimiento** (Earned Value) ya calculada.
4. El archivo original es **rico pero complejo** y poco práctico para usarlo directamente como herramienta de gestión diaria.

### Próximo paso

En el siguiente notebook (`02_Estructura_Datos_Proyecto`) vamos a definir de forma clara qué información necesitamos conservar y cómo la vamos a estructurar para construir un sistema de seguimiento ordenado y usable.